# Notebook 01: Data Setup — Three-Arm Shift Design

**Purpose**: Build the real-world data foundation for RQ1/RQ2, and the WHYSHIFT
screening that picks which real-world state pairs test which shift type.
**No training, no LLM calls. CPU only.**

## Why this notebook looks the way it does

`TABLESHIFT_AUDIT.md` found that TableShift's three named shift types are not
independently measured: its "concept shift" metric (`Δ_y|x`, a Fréchet distance
between classifier activations) correlates ρ=0.99 with its own covariate metric
(`Δ_x`) — one measure reported twice under two names. Only label shift (`Δ_y`)
independently predicts the OOD performance gap (ρ=0.73). Any claim of the form
"protocol X helps under covariate shift but hurts under concept shift" **cannot**
be tested on TableShift.

So this project uses **three arms**, each targeting one measurable shift type:

| Arm | Source | Shift tested | Why this source |
|---|---|---|---|
| **Label shift** | TableShift (`acspubcov`) | P(Y) changes | The only TableShift dataset with a strong, independent label-shift signal (`Δ_y`=0.170, our own prior-shift estimate=0.414, supervised accuracy drop=18.9% — see `TABLESHIFT_AUDIT.md` §3) |
| **Covariate + concept shift** | WHYSHIFT (ACS Income + ACS PubCov via `folktables`) | P(X) and P(Y\|X), **measured** via DISDE | Liu et al.'s DISDE decomposition (NeurIPS 2023) splits the source→target performance gap into a covariate component and a concept component for *every* state pair — so the shift type is a computed quantity, not an assumed label |
| **Controlled** | `src/data/generator.py` (synthetic) | Isolated shifts + spurious correlation, ground truth known | Full experimental control for RQ3/RQ4 |

**No corrections are imported from the prior TableShift-only analysis** (contextual
calibration necessity, verbaliser choices, token-anchoring) — those were fitted to
4 TableShift tasks where the LLM has strong semantic priors, and may not hold on
WHYSHIFT or the synthetic arm. This notebook only prepares data; NB03 observes
model behaviour on it before any correction is decided.

**Literature gap**: no published work combines LLM in-context demonstration
selection with WHYSHIFT, or with any tabular benchmark that independently labels
shift type (as of Sep 2026). Closest is Zeng et al. (2024, NeurIPS TRL Workshop),
which uses LLM *embeddings* + finetuning on WHYSHIFT, not ICL prompting.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, set_seed, resolve_path

config = load_config()
set_seed(config.seed_accuracy[0])

## Section A: WHYSHIFT — DISDE screening

`scripts/extract_whyshift_cache.py` has already cached, per task, the source
state's train/test_id split and one `test_ood_{STATE}.parquet` per candidate
target state (8 candidates each for ACS Income and ACS PubCov — see that
script's docstring for the full list and why those states were chosen).

For each task, `whyshift_loader.screen_disde_shifts` trains one quick XGBoost
baseline on the source state and calls `whyshift.degradation_decomp()` against
every candidate target — the DISDE method (Liu et al., NeurIPS 2023) that splits
the source→target accuracy drop into a covariate (P(X)) share and a concept
(P(Y|X)) share. This baseline model is a diagnostic only; it is never used in the
LLM ICL experiments themselves.

If you haven't run the extraction script yet:
```bash
pip install folktables whyshift
python scripts/extract_whyshift_cache.py
```

In [3]:
from src.data.whyshift_loader import WHYSHIFT_DATASETS, screen_disde_shifts

disde_results = {}
for task_name in WHYSHIFT_DATASETS:
    print(f'Screening {task_name}...')
    disde_results[task_name] = screen_disde_shifts(task_name)
    display(disde_results[task_name])

Screening whyshift_income...


,target_state,total_degradation,proportion_yx_shift,proportion_x_shift
0,FL,0.070886,0.870835,0.129165
1,WV,0.097683,0.774734,0.225266
2,AL,0.080836,0.746775,0.253225
3,NY,0.046586,0.727967,0.272033
4,CO,0.053536,0.708338,0.291662
5,WA,0.031086,0.696977,0.303023
6,AR,0.081079,0.636540,0.363460
7,MS,0.094129,0.628055,0.371945


Screening whyshift_pubcov...


,target_state,total_degradation,proportion_yx_shift,proportion_x_shift
0,MN,0.069417,2.545603,-1.545603
1,MA,0.103703,1.869730,-0.869730
2,CT,0.151645,1.157055,-0.157055
3,VT,0.263013,1.129293,-0.129293
4,GA,0.171661,1.046913,-0.046913
5,FL,0.166403,1.014791,-0.014791
6,NV,0.221282,0.936417,0.063583
7,AZ,0.197375,0.927116,0.072884


### Selecting pairs

Per task: the target state with the **highest** `proportion_yx_shift` is the
concept-shift-dominant pair. If any candidate has `proportion_yx_shift < 0.5`,
the lowest of those is the covariate-shift-dominant pair. WHYSHIFT's own paper
reports 87% of high-shift pairs are Y|X-dominant, so a covariate-dominant
candidate is not guaranteed to exist for every task — if none does, that's
itself a finding worth reporting, and covariate shift is then tested only on
the synthetic arm.

In [5]:
selected_pairs = []  # list of (task_name, target_state, shift_label)

for task_name, df in disde_results.items():
    concept_row = df.iloc[0]  # highest proportion_yx_shift
    selected_pairs.append((task_name, concept_row['target_state'], 'concept'))
    print(f"{task_name}: concept-dominant target = {concept_row['target_state']} "
          f"(Y|X proportion = {concept_row['proportion_yx_shift']:.3f}, "
          f"total degradation = {concept_row['total_degradation']:.3f})")

    covariate_candidates = df[df['proportion_yx_shift'] < 0.5]
    if not covariate_candidates.empty:
        covariate_row = covariate_candidates.iloc[-1]  # lowest proportion_yx among candidates
        selected_pairs.append((task_name, covariate_row['target_state'], 'covariate'))
        print(f"{task_name}: covariate-dominant target = {covariate_row['target_state']} "
              f"(Y|X proportion = {covariate_row['proportion_yx_shift']:.3f})")
    else:
        print(f"{task_name}: no covariate-dominant candidate (all Y|X proportions >= 0.5) "
              "-- covariate shift for this task is tested on the synthetic arm only.")

selected_pairs

whyshift_income: concept-dominant target = FL (Y|X proportion = 0.871, total degradation = 0.071)
whyshift_income: no covariate-dominant candidate (all Y|X proportions >= 0.5) -- covariate shift for this task is tested on the synthetic arm only.
whyshift_pubcov: concept-dominant target = MN (Y|X proportion = 2.546, total degradation = 0.069)
whyshift_pubcov: no covariate-dominant candidate (all Y|X proportions >= 0.5) -- covariate shift for this task is tested on the synthetic arm only.


[('whyshift_income', 'FL', 'concept'), ('whyshift_pubcov', 'MN', 'concept')]

## Section B: WHYSHIFT — process selected pairs

For each selected `(task, target_state)` pair: load the cached splits, select
the top-10 features by mutual information (matching the synthetic arm's
dimensionality), mode/median-impute, build a 256-row stratified demo pool, and
sample 500 ID-test / 500 OOD-test rows. Reuses the exact same generic
functions `tableshift_loader.py` already defines — nothing WHYSHIFT-specific
about feature selection, imputation, pooling, or the artifact format.

Saved dataset name: `{task_name}_{target_state}` (e.g. `whyshift_income_MS`),
so a task selected for both a concept-dominant and a covariate-dominant target
produces two distinct datasets.

In [6]:
import json

from tqdm import tqdm

from src.data.tableshift_loader import (
    load_codebook,
    select_top_features,
    impute_missing,
    build_demo_pool,
    save_dataset_artifacts,
)
from src.data.whyshift_loader import load_whyshift_splits, default_whyshift_cache_dir

whyshift_manifest = {}  # dataset_name -> shift_label

for task_name, target_state, shift_label in tqdm(selected_pairs, desc='WHYSHIFT pairs'):
    dataset_name = f'{task_name}_{target_state}'
    splits = load_whyshift_splits(task_name, target_state)
    codebook = load_codebook(Path(default_whyshift_cache_dir()) / task_name)
    feature_cols = select_top_features(splits['train'], n_features=config.generator.n_features)

    train_imputed = impute_missing(splits['train'], feature_cols)
    test_id_imputed = impute_missing(splits['test_id'], feature_cols)
    test_ood_imputed = impute_missing(splits['test_ood'], feature_cols)

    train_pool = build_demo_pool(train_imputed, config.pool_size, seed=config.seed_accuracy[0])
    test_id = test_id_imputed.sample(
        n=min(config.test_rows_id, len(test_id_imputed)), random_state=config.seed_accuracy[0]
    ).reset_index(drop=True)
    test_ood = test_ood_imputed.sample(
        n=min(config.test_rows_ood, len(test_ood_imputed)), random_state=config.seed_accuracy[0]
    ).reset_index(drop=True)

    label_tokens = [str(int(v)) for v in sorted(splits['train']['label'].unique())]

    save_dataset_artifacts(
        dataset_name=dataset_name,
        train_pool=train_pool,
        test_id=test_id,
        test_ood=test_ood,
        feature_list=feature_cols,
        label_tokens=label_tokens,
        out_root=resolve_path(config.paths.data_real),
        codebook=codebook,
    )
    whyshift_manifest[dataset_name] = shift_label
    print(f'{dataset_name} ({shift_label} shift): pool={len(train_pool)} '
          f'id={len(test_id)} ood={len(test_ood)} '
          f'base_rate_id={test_id["label"].mean():.3f} base_rate_ood={test_ood["label"].mean():.3f}')

WHYSHIFT pairs: 100%|██████████| 2/2 [00:00<00:00,  5.64it/s]

whyshift_income_FL (concept shift): pool=256 id=500 ood=500 base_rate_id=0.378 base_rate_ood=0.328
whyshift_pubcov_MN (concept shift): pool=256 id=500 ood=500 base_rate_id=0.188 base_rate_ood=0.328


## Section C: TableShift — label shift arm

`acspubcov` is the only TableShift dataset with a strong, independently-measured
label shift (see the module docstring table above). Its OOD split is a
geographic/temporal domain split (Census `DIVISION`), not a state pair — a
different shift mechanism from WHYSHIFT's `whyshift_pubcov`, even though it's
the same underlying task. That's a useful controlled comparison: same task,
two different real-world shift mechanisms.

Requires the TableShift raw cache to already exist under
`data/tableshift_raw_cache/acspubcov/` (see `scripts/extract_tableshift_cache.py`
— it must be run once from a separate, isolated environment; TableShift itself
is never installed into this project's own environment).

In [7]:
from src.data.tableshift_loader import (
    load_tableshift_splits,
    default_raw_cache_dir,
    TASK_DESCRIPTIONS,
)

TABLESHIFT_LABEL_SHIFT_DATASETS = ['acspubcov']

for dataset_name in tqdm(TABLESHIFT_LABEL_SHIFT_DATASETS, desc='TableShift (label shift)'):
    splits = load_tableshift_splits(dataset_name)
    codebook = load_codebook(Path(default_raw_cache_dir()) / dataset_name)
    feature_cols = select_top_features(splits['train'], n_features=config.generator.n_features)

    train_imputed = impute_missing(splits['train'], feature_cols)
    test_id_imputed = impute_missing(splits['test_id'], feature_cols)
    test_ood_imputed = impute_missing(splits['test_ood'], feature_cols)

    train_pool = build_demo_pool(train_imputed, config.pool_size, seed=config.seed_accuracy[0])
    test_id = test_id_imputed.sample(
        n=min(config.test_rows_id, len(test_id_imputed)), random_state=config.seed_accuracy[0]
    ).reset_index(drop=True)
    test_ood = test_ood_imputed.sample(
        n=min(config.test_rows_ood, len(test_ood_imputed)), random_state=config.seed_accuracy[0]
    ).reset_index(drop=True)

    label_tokens = [str(int(v)) for v in sorted(splits['train']['label'].unique())]

    save_dataset_artifacts(
        dataset_name=dataset_name,
        train_pool=train_pool,
        test_id=test_id,
        test_ood=test_ood,
        feature_list=feature_cols,
        label_tokens=label_tokens,
        out_root=resolve_path(config.paths.data_real),
        codebook=codebook,
    )
    print(f'{dataset_name} (label shift): pool={len(train_pool)} id={len(test_id)} ood={len(test_ood)} '
          f'base_rate_id={test_id["label"].mean():.3f} base_rate_ood={test_ood["label"].mean():.3f}')

TableShift (label shift): 100%|██████████| 1/1 [00:01<00:00,  1.72s/it]

acspubcov (label shift): pool=256 id=500 ood=500 base_rate_id=0.206 base_rate_ood=0.604


## Section D: Dataset summary table

Purely descriptive — no gates, no corrections applied here. This table is
the reference point for every downstream notebook and a candidate thesis
figure/table on its own.

In [8]:
import pandas as pd

ALL_DATASETS = {**whyshift_manifest, **{name: 'label' for name in TABLESHIFT_LABEL_SHIFT_DATASETS}}

summary_rows = []
for dataset_name, shift_label in ALL_DATASETS.items():
    dataset_dir = resolve_path(config.paths.data_real) / dataset_name
    pool = pd.read_parquet(dataset_dir / 'train_pool.parquet')
    test_id = pd.read_parquet(dataset_dir / 'test_id.parquet')
    test_ood = pd.read_parquet(dataset_dir / 'test_ood.parquet')
    feature_list = json.load(open(dataset_dir / 'feature_list.json'))

    arm = 'label' if shift_label == 'label' else 'covariate/concept (WHYSHIFT)'
    disde_proportion = None
    if dataset_name in whyshift_manifest:
        task_name, target_state = dataset_name.rsplit('_', 1)
        row = disde_results[task_name]
        match = row[row['target_state'] == target_state]
        if not match.empty:
            disde_proportion = float(match.iloc[0]['proportion_yx_shift'])

    summary_rows.append({
        'dataset': dataset_name,
        'arm': arm,
        'shift_type': shift_label,
        'n_features': len(feature_list),
        'pool_size': len(pool),
        'test_id_size': len(test_id),
        'test_ood_size': len(test_ood),
        'base_rate_id': round(test_id['label'].mean(), 3),
        'base_rate_ood': round(test_ood['label'].mean(), 3),
        'disde_proportion_yx': disde_proportion,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,dataset,arm,shift_type,n_features,pool_size,test_id_size,test_ood_size,base_rate_id,base_rate_ood,disde_proportion_yx
0,whyshift_income_FL,covariate/concept (WHYSHIFT),concept,10,256,500,500,0.378,0.328,0.870835
1,whyshift_pubcov_MN,covariate/concept (WHYSHIFT),concept,10,256,500,500,0.188,0.328,2.545603
2,acspubcov,label,label,10,256,500,500,0.206,0.604,NaN


## Section E: Serialisation examples + output manifest

One serialised demo + query per dataset, via `src/data/serialisation.py`.
Confirms the codebook renders human-readable feature names and category
labels (e.g. "sex: Female", not "SEX: 0" — the artefact present in the
TableShift-derived codebooks, see the module docstring in
`scripts/extract_whyshift_cache.py`).

In [9]:
from src.data.serialisation import serialise_row, ordered_feature_names

for dataset_name in ALL_DATASETS:
    dataset_dir = resolve_path(config.paths.data_real) / dataset_name
    pool = pd.read_parquet(dataset_dir / 'train_pool.parquet')
    feature_list = json.load(open(dataset_dir / 'feature_list.json'))
    codebook = load_codebook(dataset_dir)

    row = pool.iloc[0]
    ordered_feats = ordered_feature_names({f: row[f] for f in feature_list})
    demo_text = serialise_row({f: row[f] for f in ordered_feats}, label=str(int(row['label'])), codebook=codebook)
    query_text = serialise_row({f: row[f] for f in ordered_feats}, codebook=codebook)

    print(f'--- {dataset_name} ---')
    print(demo_text)
    print(query_text)
    print()

--- whyshift_income_FL ---
AGEP: 25.00; COW: Employee of a private for-profit company or business, or of an individual, for wages, salary, or commissions; MAR: Never married or under 15 years old; OCCP: FIN-Accountants And Auditors; POBP: Ohio/OH; RAC1P: White alone; RELP: Roomer or boarder; SCHL: Bachelor's degree; SEX: Female; WKHP: 40.00 -> 1
AGEP: 25.00; COW: Employee of a private for-profit company or business, or of an individual, for wages, salary, or commissions; MAR: Never married or under 15 years old; OCCP: FIN-Accountants And Auditors; POBP: Ohio/OH; RAC1P: White alone; RELP: Roomer or boarder; SCHL: Bachelor's degree; SEX: Female; WKHP: 40.00 ->

--- whyshift_pubcov_MN ---
AGEP: 18.00; CIT: Born in the U.S.; DEYE: No; DIS: Without a disability; DREM: No; ESR: Not in labor force; MIL: Never served in the military; NATIVITY: Native; PINCP: 14500.00; SCHL: Regular high school diploma -> 1
AGEP: 18.00; CIT: Born in the U.S.; DEYE: No; DIS: Without a disability; DREM: No; ESR: 

In [10]:
print('Output manifest — data/real/{dataset}/:')
for dataset_name, shift_label in ALL_DATASETS.items():
    dataset_dir = resolve_path(config.paths.data_real) / dataset_name
    files = sorted(p.name for p in dataset_dir.iterdir())
    print(f'  {dataset_name} ({shift_label} shift): {files}')

Output manifest — data/real/{dataset}/:
  whyshift_income_FL (concept shift): ['codebook.json', 'feature_list.json', 'label_tokens.json', 'test_id.parquet', 'test_ood.parquet', 'train_pool.parquet']
  whyshift_pubcov_MN (concept shift): ['codebook.json', 'feature_list.json', 'label_tokens.json', 'test_id.parquet', 'test_ood.parquet', 'train_pool.parquet']
  acspubcov (label shift): ['codebook.json', 'feature_list.json', 'label_tokens.json', 'test_id.parquet', 'test_ood.parquet', 'train_pool.parquet']
